# Ensemble results: confusion matrices and ML vs DESeq2 gene comparison

1. **Confusion matrices** for each model in the ensemble plus the majority-vote ensemble, using
   cross-validated predictions (every sample is predicted by a model that never saw it in training).
2. **Gene comparison**: the ensemble's consensus important genes vs DESeq2 significant genes.

Run this on your Mac with the Docker stack up. It talks to the data service at `localhost:50051`.

In [1]:
# ---- Config: edit to match your setup ----
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()            # ML_PIPELINE/ (run the notebook from there)
DATA_SERVICE_URL = "localhost:50051"  # host side; inside Docker it's data_service:50051
DATASET_ID = "symbols_filtered"       # transformed dataset the ML models train on
LABEL_COL = "Factor Value[Spaceflight]"  # condition column; set to None to auto-detect
POSITIVE_LABEL = "Space Flight"
NEGATIVE_LABEL = "Ground Control"

# Pipeline outputs for the gene comparison (results/ is mounted from the containers)
CONSENSUS_GENES_PATH = PROJECT_ROOT / "results" / "consensus_genes.csv"
DESEQ2_RESULTS_PATH = PROJECT_ROOT / "results" / "deseq2_{DATASET_ID}" / "deseq2_results.csv"
PADJ_THRESHOLD = 0.05
LOG2FC_THRESHOLD = 0.0
TOP_N_ML = None   # e.g. 20 to compare only the top-20 consensus genes

# gene_comparison.py lives in the orchestration service
sys.path.insert(0, str(PROJECT_ROOT / "orchestration_service" / "src"))
sys.path.insert(0, str(PROJECT_ROOT / "feature_importance_service" / "src"))

USE_DEMO_DATA = False  # True = synthetic data, for checking the notebook without the services

## 1. Load the dataset

In [2]:
import numpy as np
import pandas as pd

def load_from_data_service(dataset_id):
    from data_client import DataServiceClient
    client = DataServiceClient(service_url=DATA_SERVICE_URL)
    for name in ("get_dataset", "load_dataset", "fetch_dataset", "stream_dataset"):
        if hasattr(client, name):
            return getattr(client, name)(dataset_id)
    raise AttributeError("DataServiceClient has none of get_dataset/load_dataset/fetch_dataset/"
                         "stream_dataset; use whichever method feature_importance_service calls.")

def make_demo_data(n_per_group=30, n_genes=100, n_de=12, seed=0):
    rng = np.random.default_rng(seed)
    genes = [f"Gene{i:03d}" for i in range(n_genes)]
    y = np.array([NEGATIVE_LABEL] * n_per_group + [POSITIVE_LABEL] * n_per_group)
    X = rng.normal(size=(2 * n_per_group, n_genes))
    X[y == POSITIVE_LABEL, :n_de] += 1.2   # first n_de genes differ between groups
    df = pd.DataFrame(X, columns=genes)
    df[LABEL_COL or "condition"] = y
    return df

df = make_demo_data() if USE_DEMO_DATA else load_from_data_service(DATASET_ID)

if LABEL_COL is None or LABEL_COL not in df.columns:
    candidates = [c for c in df.columns if df[c].dtype == object or "factor" in c.lower()
                  or c.lower() in ("condition", "label", "target")]
    if not candidates:
        raise ValueError(f"Can't find the label column; set LABEL_COL. Columns: {list(df.columns)[:10]}...")
    LABEL_COL = candidates[0]
    print(f"Using label column: {LABEL_COL!r}")

y = df[LABEL_COL].astype(str).values
X = df.drop(columns=[LABEL_COL]).select_dtypes(include="number")
print(f"X: {X.shape[0]} samples x {X.shape[1]} genes")
print(pd.Series(y).value_counts().to_string())

ERROR:data_client:Failed to import protobuf modules: attempted relative import with no known parent package
ERROR:data_client:Make sure to regenerate data_service.proto:
ERROR:data_client:  python regenerate_data_service_proto.py


ImportError: attempted relative import with no known parent package

## 2. Confusion matrices

Edit `MODELS` to match the algorithms in your ensemble. Predictions come from stratified k-fold
`cross_val_predict`, so with ~60 samples these are honest out-of-sample numbers, not training accuracy.
The ensemble prediction is a simple majority vote across models (ties go to the first label).

In [ ]:
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, accuracy_score, balanced_accuracy_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC

MODELS = {
    "Random Forest":       RandomForestClassifier(n_estimators=300, random_state=42),
    "Extra Trees":         ExtraTreesClassifier(n_estimators=300, random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(random_state=42),
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),
    "SVC":                 make_pipeline(StandardScaler(), SVC(kernel="rbf")),
}

labels = [NEGATIVE_LABEL, POSITIVE_LABEL] if set(y) == {NEGATIVE_LABEL, POSITIVE_LABEL} else sorted(set(y))
n_splits = min(5, pd.Series(y).value_counts().min())
cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

preds = {name: cross_val_predict(model, X, y, cv=cv) for name, model in MODELS.items()}

# Majority vote across models
votes = pd.DataFrame(preds)
preds["Ensemble (majority vote)"] = votes.apply(
    lambda row: max(labels, key=lambda lab: (row == lab).sum()), axis=1).values

summary = pd.DataFrame({
    name: {"accuracy": accuracy_score(y, p), "balanced_accuracy": balanced_accuracy_score(y, p)}
    for name, p in preds.items()
}).T.round(3)
display(summary)

In [ ]:
n = len(preds)
ncols = 3
nrows = int(np.ceil(n / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.2 * ncols, 3.9 * nrows))
axes = np.atleast_1d(axes).ravel()

for ax, (name, p) in zip(axes, preds.items()):
    cm = confusion_matrix(y, p, labels=labels)
    ConfusionMatrixDisplay(cm, display_labels=labels).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(f"{name}\nacc {summary.loc[name, 'accuracy']:.2f}", fontsize=11)
    ax.tick_params(axis="x", labelrotation=20)

for ax in axes[n:]:
    ax.axis("off")

fig.suptitle(f"Cross-validated confusion matrices ({n_splits}-fold, {DATASET_ID})", fontsize=13)
fig.tight_layout()
plt.show()

## 3. ML consensus genes vs DESeq2

Set `CONSENSUS_GENES_PATH` and `DESEQ2_RESULTS_PATH` in the config cell to wherever your pipeline
writes these. The consensus file needs a gene column (and optionally a votes/score column); the DESeq2
file needs gene, `log2FoldChange` and `padj` columns.

The p-value is a hypergeometric test: how likely is an overlap this big if the ML genes were a random
draw from the same tested genes?

In [ ]:
from gene_comparison import compare_genes

if USE_DEMO_DATA:
    # Demo: rank genes by RF importance and run a t-test as a stand-in for DESeq2
    from scipy.stats import ttest_ind
    from statsmodels.stats.multitest import multipletests
    rf = RandomForestClassifier(n_estimators=300, random_state=42).fit(X, y)
    consensus = pd.Series(rf.feature_importances_, index=X.columns).nlargest(20)
    consensus = pd.DataFrame({"gene": consensus.index, "score": consensus.values})
    a, b = X[y == POSITIVE_LABEL], X[y == NEGATIVE_LABEL]
    t, pv = ttest_ind(a, b)
    deseq2 = pd.DataFrame({"gene": X.columns, "log2FoldChange": a.mean() - b.mean(),
                           "padj": multipletests(pv, method="fdr_bh")[1]})
else:
    consensus = pd.read_csv(CONSENSUS_GENES_PATH)
    deseq2 = pd.read_csv(DESEQ2_RESULTS_PATH)

cmp = compare_genes(consensus, deseq2, universe=X.columns,
                    padj_threshold=PADJ_THRESHOLD, log2fc_threshold=LOG2FC_THRESHOLD,
                    top_n_ml=TOP_N_ML)

print(f"ML consensus genes:      {cmp['n_ml']}")
print(f"DESeq2 significant genes:{cmp['n_deseq2']:>4}  (padj < {PADJ_THRESHOLD}, |log2FC| > {LOG2FC_THRESHOLD})")
print(f"Overlap:                 {cmp['n_overlap']}  (expected by chance {cmp['expected_overlap']})")
print(f"Fold enrichment:         {cmp['fold_enrichment']}")
print(f"Hypergeometric p:        {cmp['hypergeom_pvalue']:.3g}")
print(f"Jaccard:                 {cmp['jaccard']}")
print(f"\nGenes found by both: {', '.join(cmp['overlap']) or '(none)'}")

In [ ]:
display(cmp["table"].style.format({"ml_score": "{:.4g}", "log2FoldChange": "{:.2f}", "padj": "{:.2e}"}, na_rep="—"))

In [ ]:
# Volcano plot of all tested genes, colored by where each gene was found
de_all = deseq2.copy()
gcol = next(c for c in de_all.columns if c.lower() in ("gene", "gene_id", "gene_symbol", "symbol", "feature", "name"))
de_all["gene"] = de_all[gcol].astype(str)
cat = cmp["table"].set_index("gene")["category"]
de_all["category"] = de_all["gene"].map(cat).fillna("neither")
de_all["neglog10_padj"] = -np.log10(de_all["padj"].clip(lower=1e-300))

style = {
    "neither":     dict(color="#c4c4c4", s=14, label="Neither"),
    "deseq2_only": dict(color="#4c78a8", s=22, label="DESeq2 only"),
    "ml_only":     dict(color="#f58518", s=22, label="ML only"),
    "both":        dict(color="#d62728", s=40, label="Both"),
}
fig, ax = plt.subplots(figsize=(7.5, 5.5))
for c, kw in style.items():
    sub = de_all[de_all["category"] == c]
    if len(sub):
        ax.scatter(sub["log2FoldChange"], sub["neglog10_padj"], alpha=0.85, edgecolor="none",
                   label=f"{kw.pop('label')} ({len(sub)})", **kw)
for _, r in de_all[de_all["category"] == "both"].iterrows():
    ax.annotate(r["gene"], (r["log2FoldChange"], r["neglog10_padj"]), fontsize=8,
                xytext=(3, 3), textcoords="offset points")
ax.axhline(-np.log10(PADJ_THRESHOLD), ls="--", lw=0.8, color="grey")
ax.set_xlabel("log2 fold change (Space Flight vs Ground Control)")
ax.set_ylabel("-log10 adjusted p")
ax.set_title("DESeq2 results with ML consensus genes highlighted")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()